# Tuning across cavity types

Every cavity model declares **its own** tuning handles — there is no fixed list.
The tuner asks the model what it can vary, so the same `cav.tune.run(...)` call
works whether the handle is an elliptical equator radius, a pillbox radius, a gun
wall radius, or a spline control point. Here we tune four different geometries to
a target frequency and compare.

See the [tuning guide](../../tuning).

In [1]:
import os
import tempfile

import numpy as np
import pandas as pd

from cavsim2d import EllipticalCavity, Pillbox, RFGun, SplineCavity, Study
from cavsim2d.utils.style import apply_style

apply_style()

## 1. Four cavities, four handles

Each cavity is paired with a geometry handle and a target frequency. A model's
`tune_variables()` lists everything it is willing to vary.

In [2]:
GUN = {'geometry': {'y1': 1.5e-2, 'R2': 3e-2, 'T2': np.deg2rad(45), 'L3': 24e-2,
                    'R4': 5e-2, 'L5': 11e-2, 'R6': 6e-2, 'L7': 19e-2, 'R8': 4e-2,
                    'T9': np.deg2rad(8), 'R10': 3e-2, 'T10': np.deg2rad(40),
                    'L11': 5e-2, 'R12': 3e-2, 'L13': 3e-2, 'R14': 3e-2, 'x': 1e-2}}
SPLINE = {'geometry': {'p0': [0, 35], 'p1': [0, 70], 'p2': [30, 103],
                       'p3': [85, 103], 'p4': [115, 70], 'p5': [115, 35]}}

jobs = [
    ('elliptical', EllipticalCavity(1, [42, 42, 12, 19, 35, 57.7, 108.0],
                                    [42, 42, 12, 19, 35, 57.7, 108.0],
                                    [42, 42, 12, 19, 35, 57.7, 108.0], beampipe='none'),
     'Req',  1300.0),
    ('pillbox',    Pillbox(1, [100, 100, 20, 0, 50], beampipe='none'),   'Req',  1300.0),
    ('rf gun',     RFGun(GUN),                                           'R6',    210.0),
    ('spline',     SplineCavity(SPLINE),                                 'p2_r', 1300.0),
]
for name, cav, handle, target in jobs:
    print(f"{name:11s} handle {handle:5s} -> {target:7.1f} MHz")

elliptical  handle Req   ->  1300.0 MHz
pillbox     handle Req   ->  1300.0 MHz
rf gun      handle R6    ->   210.0 MHz
spline      handle p2_r  ->  1300.0 MHz


## 2. Tune each to its target

The call is identical for every model; only the handle name changes. We record the
starting frequency, the frequency the tuner reports, and the frequency of an
**independent** re-solve of the tuned cavity — the strict check that it really
landed on target.

In [3]:
rows = []
for name, cav, handle, target in jobs:
    study = Study(os.path.join(tempfile.mkdtemp(), name))
    study.add_cavity(cav, name.replace(' ', '_'))
    cav.eigenmode.run({'polarisation': 'monopole', 'mesh_config': {'h': 6, 'p': 2}})
    f0 = cav.eigenmode.qois['freq [MHz]']

    study.run_tune({'freqs': target, 'cell_type': {'mid-cell': handle},
                    'processes': 1, 'rerun': True, 'maxiter': 30,
                    'eigenmode_config': {'mesh_config': {'h': 6, 'p': 2}}})
    f_tuner = cav.tune.qois['mid-cell']['FREQ']

    cav.tuned.eigenmode.run({'polarisation': 'monopole', 'mesh_config': {'h': 6, 'p': 2}})
    f_res = cav.tuned.eigenmode.qois['freq [MHz]']
    rows.append({'cavity': name, 'handle': handle, 'start [MHz]': round(f0, 1),
                 'target [MHz]': target, 'tuner [MHz]': round(f_tuner, 2),
                 're-solved [MHz]': round(f_res, 2)})

ERROR:: Done Tuning Cavity pillbox [mid-cell: Req]: Failed: Accuracy of 1.00e-04 could not be reached. Accuracy of 3.15e-04 reached.


## 3. Summary

Every geometry reaches its target, and the independent re-solve of `cav.tuned`
agrees with the requested frequency — the tuned cavity really sits on target.
(The small re-solve offsets are mesh-resolution differences, not tuning error.)

In [4]:
pd.DataFrame(rows).set_index('cavity')

,handle,start [MHz],target [MHz],tuner [MHz],re-solved [MHz]
cavity,,,,,
elliptical,Req,1236.9,1300.0,1300.0,1300.04
pillbox,Req,1160.4,1300.0,1300.0,1300.73
rf gun,R6,213.1,210.0,210.0,210.01
spline,p2_r,1535.6,1300.0,1300.0,1301.09
